# Ficha automática de mercado y entorno — Venture One SAS

**Reto 13 · EstudIA · Análisis de datos · Capa 2 · Nivel básico**

Objetivo: generar, para cada inmueble de prueba, una ficha JSON con:
1. **Mercado:** precio/m² de comparables en el mismo estado y una regla de confianza.
2. **Entorno:** estación de transporte más cercana y equipamientos dentro del radio de búsqueda.
3. **Norma:** tratamiento del POT (cuando la capa esté disponible).
4. **Calidad:** duplicados, datos inválidos y datos faltantes.

Lea antes de empezar:
- `docs/diccionario_datos.md` — columnas y calidad de cada archivo.
- `config/parametros.yaml` — radio, estados y regla de confianza. **Ningún número va fijo en el código.**
- `docs/plantilla_ficha.md` y `ejemplos/ficha_ejemplo.json` — formato de salida.

Reglas del reto: use solo los datos entregados. No haga scraping de portales. No use datos personales.

In [39]:
# 1. Preparar el entorno: clona el repositorio en Colab y se ubica en su raíz.
import os, subprocess, sys

REPO_URL = "https://github.com/solhecoo/ventureone-reto-ficha.git"
CARPETA = "ventureone-reto-ficha"

if "google.colab" in sys.modules and not os.path.exists(CARPETA):
    subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
if os.path.exists(CARPETA):
    os.chdir(CARPETA)
elif os.path.exists("../config/parametros.yaml"):
    os.chdir("..")  # ejecución local desde notebooks/
assert os.path.exists("config/parametros.yaml"), "No se encontró la raíz del repositorio"
print("Raíz del repositorio:", os.getcwd())

Raíz del repositorio: /content/ventureone-reto-ficha/ventureone-reto-ficha/ventureone-reto-ficha


In [40]:
# 2. Cargar parámetros y datos.
import csv, json
import yaml

PARAMS = yaml.safe_load(open("config/parametros.yaml", encoding="utf-8"))
ANUNCIOS = list(csv.DictReader(open("data/anuncios_muestra.csv", encoding="utf-8")))
INMUEBLES = list(csv.DictReader(open("data/inmuebles_prueba.csv", encoding="utf-8")))
TRANSPORTE = json.load(open("data/geo/transporte.geojson", encoding="utf-8"))
EQUIPAMIENTOS = json.load(open("data/geo/equipamientos.geojson", encoding="utf-8"))
POT = json.load(open(PARAMS["norma"]["archivo_pot"], encoding="utf-8")) if os.path.exists(PARAMS["norma"]["archivo_pot"]) else None

print(f"Anuncios: {len(ANUNCIOS)} · Inmuebles de prueba: {len(INMUEBLES)}")
print(f"Estaciones: {len(TRANSPORTE['features'])} · Equipamientos: {len(EQUIPAMIENTOS['features'])} · OSM {TRANSPORTE['metadata']['fecha_osm']}")
print("Capa POT:", "cargada" if POT else "pendiente")

Anuncios: 117 · Inmuebles de prueba: 11
Estaciones: 42 · Equipamientos: 2714 · OSM 2026-11-13T23:03:06Z
Capa POT: pendiente


## Exploración

Revise la calidad antes de calcular: ¿cuántos `id_anuncio` se repiten? ¿Qué variantes tiene `estado`?
¿Qué columnas están vacías? Registre lo que encuentre: la ficha debe declarar los datos faltantes.

In [41]:
import pandas as pd

df = pd.read_csv("data/anuncios_muestra.csv")
display(df.head())
print("Filas por zona:\n", df["zona"].value_counts(), "\n")
print("Variantes de estado:\n", df["estado"].value_counts(dropna=False), "\n")
print("id_anuncio repetidos:", df["id_anuncio"].dropna().duplicated().sum())
print("Vacíos por columna:\n", df.isna().sum())

,id_anuncio,fuente,fecha,zona,municipio,barrio,area_m2,precio,habitaciones,banos,parqueadero,estado,tipo_precio,origen
0,fincaraiz:10710482,fincaraiz,NaN,castellana_laureles,medellin,la castellana,94.0,610000000,3.0,2.0,1,REMODELADO,pedido,factibilidad_castellana.xlsx:SECTORIAL:2
1,fincaraiz:193287983,fincaraiz,NaN,castellana_laureles,medellin,la castellana,99.0,690000000,3.0,2.0,2,REMODELADO,pedido,factibilidad_castellana.xlsx:SECTORIAL:3
2,fincaraiz:192780858,fincaraiz,NaN,castellana_laureles,medellin,laureles,92.0,497000000,3.0,2.0,-,REMODELADO,pedido,factibilidad_castellana.xlsx:SECTORIAL:4
3,fincaraiz:192971383,fincaraiz,NaN,castellana_laureles,medellin,la castellana,91.0,590000000,3.0,3.0,1,HABITABLE POR REMODELAR,pedido,factibilidad_castellana.xlsx:SECTORIAL:5
4,fincaraiz:193663348,fincaraiz,NaN,castellana_laureles,medellin,santa teresita,89.0,580000000,4.0,3.0,1,HABITABLE POR REMODELAR,pedido,factibilidad_castellana.xlsx:SECTORIAL:6


Filas por zona:
 zona
buenos_aires_el_salvador    82
ditaires_itagui             19
castellana_laureles         16
Name: count, dtype: int64 

Variantes de estado:
 estado
NaN                           25
Remodelar                     20
OK                            16
REMODELADA                    13
REMODELADO                    10
Vivir                          9
HABITABLE POR REMODELAR        6
remodelar                      5
Reformar                       3
SIN REMODELAR (OBRA NEGRA)     3
SIN REMODELAR                  2
Ok                             2
REMODELADA ?                   1
VIVIR                          1
vivir                          1
Name: count, dtype: int64 

id_anuncio repetidos: 11
Vacíos por columna:
 id_anuncio       50
fuente            0
fecha           117
zona              0
municipio         0
barrio           25
area_m2           1
precio            0
habitaciones      5
banos             9
parqueadero      82
estado           25
tipo_precio       0

## Su solución

Complete las funciones. Cada una recibe `PARAMS` para leer las reglas.
La función `haversine_m` se entrega lista.

In [42]:
import math
import sys
from datetime import date
from pathlib import Path

# Usa las funciones de referencia de src/ficha (ver README: carpeta de los organizadores).
Ruta_SRC = str(Path("src").resolve())
if Ruta_SRC not in sys.path:
    sys.path.insert(0, Ruta_SRC)

from ficha import entorno as _entorno
from ficha import limpieza as _limpieza
from ficha import mercado as _mercado
from ficha.ficha import calidad_zona, datos_faltantes
from ficha.norma import bloque_norma

# Esta celda usa las variables de la celda 2 del notebook:
# PARAMS, ANUNCIOS, INMUEBLES, TRANSPORTE, EQUIPAMIENTOS y POT.


def haversine_m(lat1, lon1, lat2, lon2):
    """Distancia en metros entre dos puntos en grados decimales."""
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = p2 - p1, math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * 6_371_008.8 * math.asin(math.sqrt(a))


def preparar_inmueble(inmueble, params):
    """Convierte el texto del CSV a número y agrega el estado canónico."""
    return {
        **inmueble,
        "area_m2": float(inmueble["area_m2"]),
        "lat": float(inmueble["lat"]),
        "lon": float(inmueble["lon"]),
        "estado_canonico": _limpieza.normalizar_estado(inmueble["estado"], params["estados"]),
    }


# Paso 4: limpieza
def limpiar_anuncios(anuncios, params):
    """Normaliza estados, elimina duplicados y marca los anuncios inválidos.

    Devuelve {"crudos": todas las filas preparadas, "anuncios": filas únicas con
    `motivo_invalido`}. Esa forma la esperan calidad_zona y datos_faltantes de src.
    """
    crudos = [_limpieza.preparar(a, params) for a in anuncios]
    unicos = _limpieza.deduplicar(crudos, params)
    for a in unicos:
        a["motivo_invalido"] = _limpieza.motivo_invalido(a, params)
    return {"crudos": crudos, "anuncios": unicos}


# Paso 5: mercado
def bloque_mercado(inmueble, anuncios_limpios, params):
    """Comparables, estadísticas de precio/m² y nivel de confianza."""
    return _mercado.bloque_mercado(inmueble, anuncios_limpios["anuncios"], params)


# Paso 6: entorno
def bloque_entorno(inmueble, transporte, equipamientos, params):
    """Estación más cercana y conteos dentro del radio."""
    return _entorno.bloque_entorno(
        inmueble["lat"], inmueble["lon"], inmueble["precision_coordenadas"],
        transporte["features"], equipamientos["features"], params["entorno"],
    )


_CACHE = {}


def generar_ficha(inmueble):
    """Arma la ficha completa con el formato de ejemplos/ficha_ejemplo.json."""
    if "limpios" not in _CACHE:
        _CACHE["limpios"] = limpiar_anuncios(ANUNCIOS, PARAMS)
    limpios = _CACHE["limpios"]
    inm = preparar_inmueble(inmueble, PARAMS)
    mercado = bloque_mercado(inm, limpios, PARAMS)
    return {
        "id_inmueble": inm["id_inmueble"],
        "inmueble": {
            "direccion": inm["direccion"] or None,
            "barrio": inm["barrio"],
            "municipio": inm["municipio"],
            "zona": inm["zona"],
            "area_m2": inm["area_m2"],
            "estado_texto": inm["estado"],
            "estado": inm["estado_canonico"],
            "coordenadas": {"lat": inm["lat"], "lon": inm["lon"],
                            "precision": inm["precision_coordenadas"],
                            "fuente": inm["fuente_coordenadas"]},
        },
        "mercado": mercado,
        "calidad_datos_zona": calidad_zona(inm["zona"], limpios),
        "entorno": bloque_entorno(inm, TRANSPORTE, EQUIPAMIENTOS, PARAMS),
        "norma": bloque_norma(inm["lat"], inm["lon"], inm["municipio"], POT, PARAMS["norma"]),
        "datos_faltantes": datos_faltantes(inm, mercado, limpios),
        "metadatos": {
            "version_parametros": PARAMS["version"],
            "fecha_generacion": date.today().isoformat(),
            "fecha_capas_osm": TRANSPORTE["metadata"]["fecha_osm"],
            "fuentes": ["data/anuncios_muestra.csv", "OpenStreetMap (ODbL)"]
                       + ([PARAMS["norma"]["archivo_pot"]] if POT else []),
        },
    }

In [43]:
# Generar una ficha por inmueble en salidas/participante/
os.makedirs("salidas/participante", exist_ok=True)
for inmueble in INMUEBLES:
    ficha = generar_ficha(inmueble)
    with open(f"salidas/participante/{inmueble['id_inmueble']}.json", "w", encoding="utf-8") as f:
        json.dump(ficha, f, ensure_ascii=False, indent=2)
print("Fichas generadas:", sorted(os.listdir("salidas/participante")))

Fichas generadas: ['INM-01.json', 'INM-02.json', 'INM-03.json', 'INM-04.json', 'INM-05.json', 'INM-06.json', 'INM-07.json', 'INM-08.json', 'INM-09.json', 'INM-10.json', 'INM-11.json']


## Validación contra el cálculo manual

`validacion/calculo_manual.md` calcula a mano INM-04, INM-06 e INM-09.
La celda compara sus fichas con esos valores y lista cada diferencia.

In [44]:
resultado = subprocess.run([sys.executable, "validacion/validar.py", "salidas/participante"], capture_output=True, text=True)
print(resultado.stdout or resultado.stderr)
print("✅ Coincide con el cálculo manual" if resultado.returncode == 0 else "❌ Revise las diferencias")

✓ INM-04
✓ INM-06
✓ INM-09

✅ Coincide con el cálculo manual
